# 1단계: TMM (Transfer Matrix Method) — AR 코팅 위상 계산

## 이 노트북에서 하는 일
스마트폰 유리 위의 **AR 코팅(반사방지막)**이 빛의 위상을 얼마나 왜곡하는지 계산합니다.

- **Gorilla DX** (4층짜리 AR 코팅): 반사율 R < 1%
- **Gorilla DX+** (8층짜리 AR 코팅): 반사율 R < 0.4%

빛이 비스듬히 들어올수록 위상 왜곡이 커지는 것을 수치와 그래프로 확인합니다.

---
**TMM이 뭔가요?**  
여러 겹으로 쌓인 얇은 막(다층막)에서 빛이 어떻게 투과/반사되는지 계산하는 방법입니다.  
각 층을 행렬(숫자표)로 표현하고, 전체를 곱해서 최종 결과를 구합니다.

**Δφ(위상 왜곡)란?**  
빛이 수직(0°)으로 들어올 때와 비교해서, 비스듬히 들어올 때 위상이 얼마나 달라지는지를 나타냅니다.  
`Δφ(θ) = ∠t(θ) − ∠t(0°)`

## 0. 필요한 도구 불러오기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import json

# 한글 폰트 설정
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

print('✓ 도구 불러오기 완료')

## 1. AR 코팅 층 구조 정의

### 원리
SiO₂(낮은 굴절률, 1.46)와 TiO₂(높은 굴절률, 2.35)를 번갈아 쌓아서 반사를 최소화합니다.  
각 층의 두께는 빛의 파장(850nm)에 맞춰 최적화되어 있습니다.

> **주의:** Gorilla DX/DX+의 정확한 층 설계는 Corning社 영업비밀입니다.  
> 여기서는 동일한 물리적 특성(T>99%, 각도에 따른 위상 변화)을 재현하는 대표 파라미터를 사용합니다.

In [ ]:
# 센싱 파장: 850 nm (NIR, 근적외선)
wavelength = 850e-9   # 단위: 미터

# 굴절률 정의
n_SiO2  = 1.46    # SiO₂ (저굴절률 재료)
n_TiO2  = 2.35    # TiO₂ (고굴절률 재료)
n_air   = 1.00    # 공기
n_glass = 1.52    # Cover Glass

# ── Gorilla DX (4층): 수치 최적화 파라미터 ──
# T > 99% @ θ=0°,  Δφ ≈ -7.5° @ θ=30°
gorilla_DX = [
    (n_SiO2,  49.5e-9),  # 층1 (공기 쪽): SiO₂
    (n_TiO2,  19.9e-9),  # 층2: TiO₂
    (n_SiO2,  29.6e-9),  # 층3: SiO₂
    (n_TiO2, 130.4e-9),  # 층4 (유리 쪽): TiO₂
]

# ── Gorilla DX+ (8층): 수치 최적화 파라미터 ──
# T > 99% @ θ=0°,  Δφ ≈ -11.9° @ θ=30°
gorilla_DX_plus = [
    (n_SiO2,  38.1e-9),  # 층1 (공기 쪽): SiO₂
    (n_TiO2,  30.1e-9),  # 층2: TiO₂
    (n_SiO2,  43.5e-9),  # 층3: SiO₂
    (n_TiO2,  63.7e-9),  # 층4: TiO₂
    (n_SiO2,   0.8e-9),  # 층5: SiO₂
    (n_TiO2,  25.6e-9),  # 층6: TiO₂
    (n_SiO2,   8.1e-9),  # 층7: SiO₂
    (n_TiO2, 190.8e-9),  # 층8 (유리 쪽): TiO₂
]

print(f'Gorilla DX  층 수: {len(gorilla_DX)}층')
print(f'Gorilla DX+ 층 수: {len(gorilla_DX_plus)}층')
print(f'센싱 파장  : {wavelength*1e9:.0f} nm')

## 2. TMM 핵심 함수 구현

**TMM의 원리:**
1. 각 층에서 빛의 전파를 2×2 행렬로 표현
2. 모든 층의 행렬을 순서대로 곱함 → 전체 시스템 행렬
3. 이 행렬에서 투과율(T)과 위상(Δφ)을 계산

**위상 왜곡 정의:**
$$\Delta\phi(\theta) = \angle t(\theta) - \angle t(0°)$$
θ=0° 기준의 **상대 위상**을 사용합니다. θ=0°에서 Δφ = 0이 됩니다.

In [ ]:
def tmm_compute(layers, n_in, n_out, theta_deg, wavelength):
    """
    Transfer Matrix Method (TMM) 계산
    
    입력:
        layers     : [(굴절률, 두께_m), ...] 층 목록 (공기 쪽 → 유리 쪽 순서)
        n_in       : 입사측 굴절률 (공기 = 1.0)
        n_out      : 출사측 굴절률 (유리 = 1.52)
        theta_deg  : 입사각 (도)
        wavelength : 파장 (미터)
    
    반환:
        T_power    : 강도 투과율 (0~1)
        t_complex  : 복소 투과 계수
    """
    theta_rad = np.radians(theta_deg)
    k0 = 2 * np.pi / wavelength
    
    # 스넬의 법칙: 수평 파수 kx는 모든 층에서 동일
    kx = n_in * np.sin(theta_rad)   # (정규화된 수평 파수)
    
    # 초기 행렬: 단위행렬
    M = np.eye(2, dtype=complex)
    
    for (n_j, d_j) in layers:
        # j번째 층에서의 수직 방향 굴절 계수
        cos_j = np.sqrt(complex(n_j**2 - kx**2))
        
        # 위상 두께 δ = k₀ · n_j · cos(θ_j) · d_j
        delta_j = k0 * cos_j * d_j
        
        # TE(s) 편광 임피던스 p_j = n_j·cos(θ_j) = cos_j
        p_j = cos_j
        
        # j번째 층의 2×2 전달 행렬
        M_j = np.array([
            [ np.cos(delta_j),           -1j * np.sin(delta_j) / p_j ],
            [ -1j * p_j * np.sin(delta_j), np.cos(delta_j)           ]
        ], dtype=complex)
        
        M = M @ M_j  # 누적 곱
    
    # 입사측/출사측 임피던스
    p_in  = np.sqrt(complex(n_in**2  - kx**2))
    p_out = np.sqrt(complex(n_out**2 - kx**2))
    
    # 복소 투과 계수 t
    denom = (M[0,0] + M[0,1] * p_out) * p_in + (M[1,0] + M[1,1] * p_out)
    t_complex = 2 * p_in / denom
    
    # 강도 투과율 T = |t|² · Re(p_out) / Re(p_in)
    T_power = (abs(t_complex)**2) * p_out.real / p_in.real
    
    return float(T_power), t_complex


def compute_phase_table(layers, n_in, n_out, theta_list, wavelength):
    """모든 각도에 대해 투과율과 상대 위상 Δφ를 계산"""
    T_arr    = np.zeros(len(theta_list))
    phi_arr  = np.zeros(len(theta_list))  # 절대 위상 (라디안)
    
    for i, theta in enumerate(theta_list):
        T_arr[i], t = tmm_compute(layers, n_in, n_out, theta, wavelength)
        phi_arr[i]  = np.angle(t)         # 복소 위상
    
    # 상대 위상: Δφ(θ) = φ(θ) - φ(0°)  [도 단위]
    dphi_arr = np.degrees(phi_arr - phi_arr[0])
    
    return T_arr, dphi_arr


print('✓ TMM 함수 정의 완료')

# 동작 확인
T0, t0 = tmm_compute(gorilla_DX, n_air, n_glass, theta_deg=0, wavelength=wavelength)
print(f'[확인] Gorilla DX θ=0°: T = {T0*100:.1f}%,  ∠t = {np.degrees(np.angle(t0)):.1f}°')

## 3. 각도별 위상 테이블 계산

In [ ]:
# 입사각 범위: 0° ~ 40° (0.5° 간격)
theta_list = np.arange(0, 40.5, 0.5)

# DX / DX+ 각각 계산
T_DX,  dphi_DX  = compute_phase_table(gorilla_DX,      n_air, n_glass, theta_list, wavelength)
T_DXp, dphi_DXp = compute_phase_table(gorilla_DX_plus, n_air, n_glass, theta_list, wavelength)

# ── 주요 각도 결과 표 ──
print(f'{"입사각θ":^6} │ {"DX 투과율":^8} │ {"DX Δφ":^9} │ {"DX+ 투과율":^9} │ {"DX+ Δφ":^9} │ PSF 영향')
print('───────┼──────────┼───────────┼───────────┼───────────┼──────────────')

key_angles = [0, 10, 15, 20, 30, 35]
effects    = ['없음 (대칭)', '미미함', '약한 비대칭', '중간 비대칭', '★ 심각한 비대칭', '★★ 더 심각']

for theta, effect in zip(key_angles, effects):
    idx = np.argmin(np.abs(theta_list - theta))
    print(f'  {theta:3d}°  │  {T_DX[idx]*100:5.1f}%   │  {dphi_DX[idx]:+7.2f}°  │  {T_DXp[idx]*100:5.1f}%   │  {dphi_DXp[idx]:+7.2f}°  │ {effect}')

## 4. 결과 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('AR 코팅 (Gorilla DX / DX+) — 입사각별 특성  λ=850nm', fontsize=14, fontweight='bold')

# ── 왼쪽: 투과율 ──
ax = axes[0]
ax.plot(theta_list, T_DX  * 100, 'b-',  lw=2, label='Gorilla DX (4층)')
ax.plot(theta_list, T_DXp * 100, 'r--', lw=2, label='Gorilla DX+ (8층)')
ax.axvline(30, color='gray', ls=':', alpha=0.7, label='θ=30° 기준선')
ax.set_xlabel('입사각 θ (°)', fontsize=12)
ax.set_ylabel('강도 투과율 T (%)', fontsize=12)
ax.set_title('투과율 vs 입사각', fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(0, 40)
ax.grid(True, alpha=0.3)

# ── 오른쪽: 상대 위상 Δφ ──
ax = axes[1]
ax.plot(theta_list, dphi_DX,  'b-',  lw=2, label='Gorilla DX (4층)')
ax.plot(theta_list, dphi_DXp, 'r--', lw=2, label='Gorilla DX+ (8층)')
ax.axvline(30, color='gray', ls=':', alpha=0.7)
ax.axhline(0,  color='black', ls='-', alpha=0.2, lw=0.8)

# θ=30° 값 표시
idx30 = np.argmin(np.abs(theta_list - 30))
ax.annotate(f'DX: {dphi_DX[idx30]:+.1f}°',
            xy=(30, dphi_DX[idx30]),
            xytext=(33, dphi_DX[idx30] + 0.5),
            fontsize=10, color='blue', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))
ax.annotate(f'DX+: {dphi_DXp[idx30]:+.1f}°',
            xy=(30, dphi_DXp[idx30]),
            xytext=(33, dphi_DXp[idx30] - 1.5),
            fontsize=10, color='red', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax.fill_between(theta_list, dphi_DX,  0, alpha=0.1, color='blue')
ax.fill_between(theta_list, dphi_DXp, 0, alpha=0.1, color='red')
ax.set_xlabel('입사각 θ (°)', fontsize=12)
ax.set_ylabel('상대 위상 왜곡 Δφ (°)', fontsize=12)
ax.set_title('위상 왜곡 Δφ vs 입사각  ← PSF 비대칭의 원인', fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(0, 40)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('TMM_결과.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 그래프 저장 완료: TMM_결과.png')

## 5. 위상 테이블 파일로 저장

다음 단계(ASM, PINN)에서 이 데이터를 불러와서 사용합니다.

In [ ]:
phase_table = {
    'wavelength_nm'        : 850,
    'description'          : 'Gorilla DX/DX+ 대표 파라미터 (상대 위상 기준)',
    'theta_deg'            : theta_list.tolist(),
    'DX_T'                 : T_DX.tolist(),
    'DX_delta_phi_deg'     : dphi_DX.tolist(),
    'DXp_T'                : T_DXp.tolist(),
    'DXp_delta_phi_deg'    : dphi_DXp.tolist(),
}

with open('tmm_phase_table.json', 'w', encoding='utf-8') as f:
    json.dump(phase_table, f, indent=2, ensure_ascii=False)

print('✓ 위상 테이블 저장: tmm_phase_table.json')
print(f'  데이터 수: {len(theta_list)}개 각도')

# 핵심 결과 요약
print('\n' + '='*55)
print('  TMM 핵심 결과 요약')
print('='*55)
for theta in [0, 15, 30, 35]:
    idx = np.argmin(np.abs(theta_list - theta))
    print(f'  θ={theta:2d}°:  DX T={T_DX[idx]*100:.1f}%  Δφ={dphi_DX[idx]:+.1f}°  |  '
          f'DX+ T={T_DXp[idx]*100:.1f}%  Δφ={dphi_DXp[idx]:+.1f}°')
print('='*55)
print('\n  → 이 위상 왜곡이 지문 이미지 비대칭(PSF skewness)의 근본 원인')
print('  → 다음 단계: 02_ASM_유리전파.ipynb')

## 6. 복소 입사 필드 U_in 구성

TMM 결과를 이용해 PINN에 입력될 **복소 입사 필드**를 구성합니다.

$$U_{in}(x; \theta) = \sqrt{T(\theta)} \cdot G(x; \sigma_0) \cdot e^{i(k_x x + \Delta\phi(\theta))}$$

- $G(x; \sigma_0)$: 광원 가우시안 형태 (σ₀ = 100μm)
- $k_x = k_0 \sin(\theta)$: 기울어진 입사
- $\Delta\phi$: AR 코팅 위상 왜곡 ← **PSF 비대칭 원인**

In [ ]:
def make_U_in(theta_deg, coating='DX', sigma0=100e-6, N=512):
    """
    복소 입사 필드 U_in(x) 생성
    
    반환: x (위치 배열, 미터), U_in (복소 필드), T (투과율), delta_phi (위상 왜곡, 도)
    """
    layers = gorilla_DX if coating == 'DX' else gorilla_DX_plus
    
    # θ=0° 기준 위상 계산
    _, t0 = tmm_compute(layers, n_air, n_glass, 0,         wavelength)
    T,  t  = tmm_compute(layers, n_air, n_glass, theta_deg, wavelength)
    
    delta_phi_deg = np.degrees(np.angle(t) - np.angle(t0))  # 상대 위상
    delta_phi_rad = np.radians(delta_phi_deg)
    
    # 공간 좌표: ±400 μm
    x = np.linspace(-400e-6, 400e-6, N)
    
    # 파수
    k0_val = 2 * np.pi / wavelength
    kx     = k0_val * np.sin(np.radians(theta_deg))
    
    # 가우시안 광원
    G = np.exp(-x**2 / (2 * sigma0**2))
    
    # 복소 입사 필드
    U_in = np.sqrt(T) * G * np.exp(1j * (kx * x + delta_phi_rad))
    
    return x, U_in, T, delta_phi_deg


# θ=0° vs θ=30° 비교 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('복소 입사 필드 U_in: θ=0° (대칭) vs θ=30° (위상 왜곡)', fontsize=13, fontweight='bold')

for col, theta in enumerate([0, 30]):
    x, U_in, T, dphi = make_U_in(theta, coating='DX')
    x_um = x * 1e6  # 미터 → μm
    
    # 진폭
    ax = axes[0, col]
    ax.plot(x_um, np.abs(U_in), 'b-', lw=1.5)
    ax.set_title(f'θ={theta}°  진폭 |U_in|  (T={T*100:.1f}%)', fontsize=11)
    ax.set_xlabel('위치 x (μm)')
    ax.set_ylabel('진폭')
    ax.set_xlim(-300, 300)
    ax.grid(True, alpha=0.3)
    
    # 위상
    ax = axes[1, col]
    phase = np.degrees(np.unwrap(np.angle(U_in)))
    ax.plot(x_um, phase, 'r-', lw=1.5)
    ax.set_title(f'θ={theta}°  위상 ∠U_in  (Δφ={dphi:+.1f}° 추가됨)', fontsize=11)
    ax.set_xlabel('위치 x (μm)')
    ax.set_ylabel('위상 (°)')
    ax.set_xlim(-300, 300)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('U_in_비교.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 그래프 저장: U_in_비교.png')

## ✅ 1단계 완료 요약

| 완료 항목 | 결과 |
|----------|------|
| TMM 함수 | `tmm_compute()` — 복소 투과 계수 계산 |
| 위상 테이블 | θ=0°~40°, DX/DX+ 상대 위상 계산 |
| 파일 저장 | `tmm_phase_table.json` |
| 입사 필드 | `make_U_in()` — 복소 입사 필드 생성 |

### 핵심 물리 확인
- **θ=0°**: Δφ = 0° → PSF 대칭 (정상)
- **θ=30°**: Δφ 증가 → PSF **비대칭** 발생
- DX+(8층)이 DX(4층)보다 각도 의존성이 더 강함

### 다음 단계
**`02_ASM_유리전파.ipynb`** — Cover Glass 550μm를 빛이 통과하는 계산 (Angular Spectrum Method)